In [1]:
import os
import regex
import json
import datetime, pytz
import speech_recognition as sr

from langchain_mistralai import ChatMistralAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import ToolMessage, HumanMessage, SystemMessage, AIMessage
from langchain.tools import tool, StructuredTool

from db_search import get_info

from text_to_speech import Text2Speech

In [2]:
import spacy
from sentence_transformers import SentenceTransformer, util

In [3]:
llm = ChatMistralAI(
    model="mistral-large-latest",
    temperature=0,
    max_retries=2,
    api_key="suFEUjmnfvT2AQSF5b6MQJ9NyZwDLsqG"
)

In [4]:
speaker = Text2Speech()

 > tts_models/multilingual/multi-dataset/xtts_v2 is already downloaded.
 > Using model: xtts


B:\Software\Anaconda\envs\voicecloning5\lib\site-packages\TTS\tts\layers\xtts\xtts_manager.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.speakers = torch.load(spe

In [5]:
nlp = spacy.load("en_core_web_sm")
model = SentenceTransformer("all-MiniLM-L6-v2")

In [6]:
intent_examples = {
    "greet": ["Hello", "Hi", "Hey", "Good morning", "Good evening", "Hi there", "Hello, is anyone there?", "Hey, can you help me?"],
    "goodbye": ["Bye", "Goodbye", "See you later", "Talk to you soon", "Have a nice day", "See you", "Thanks", "Thank you so much"],
    "loan_details": ["Can you tell me my loan details?", "What is my outstanding loan balance?", "How much loan amount is left to pay?", "I want to know my loan status", "Provide my loan details", "How much do I owe?"],
    "payment_due": ["When is my next payment due?", "Tell me my EMI due date", "What is my next EMI date?", "I need my loan payment schedule", "Can you remind me of my payment date?"],
    "make_payment": ["I want to pay my EMI", "How can I pay my loan?", "Send me the payment link", "Can I pay my EMI online?", "Where can I make the payment?"],
    "grace_period": ["Can I get an extension on my loan payment?", "I need a grace period", "Can I delay my EMI?", "I need some time to make my payment", "Is there a penalty for late payment?"],
    "financial_difficulty": ["I am facing financial issues", "I am unable to pay the loan right now", "My salary is delayed, I need help", "Can I restructure my loan?", "I need more time to make the payment"],
    "interest_rate": ["What is my loan interest rate?", "Can you tell me the interest rate on my loan?", "Has my interest rate changed?", "How is my interest calculated?", "I need details on my loan interest"],
    "late_fees": ["What are the penalties for late payment?", "Will I be charged a fee if I miss my EMI?", "How much is the late fee?", "What happens if I don’t pay on time?", "Explain the overdue charges"],
    "loan_tenure": ["What is the duration of my loan?", "How many EMIs are left?", "How long do I have to repay the loan?", "When will my loan be fully paid?", "Tell me about my repayment period"],
    "dispute_transaction": ["I see an incorrect charge on my loan statement", "My loan balance is wrong", "I was charged extra, please check", "I have a dispute regarding my loan", "Why was I charged more than usual?"],
    "confirm_payment": ["Did you receive my last EMI payment?", "Has my loan payment been processed?", "I made the payment, please confirm", "My money has been deducted but loan not updated", "Can you check my last payment status?"],
    "loan_preclosure": ["Can I pay off my loan early?", "What is the preclosure process?", "How much do I need to pay to close my loan?", "Can I settle my loan early?", "What are the charges for early loan repayment?"],
    "contact_support": ["I need to speak with a customer representative", "Can I talk to an agent?", "Provide customer support details", "I need help from a loan officer", "Who can assist me with my loan issue?"],
    "assistance": ["I need help with my loan", "Can you assist me?", "I need guidance on loan repayment", "Can someone help me with my account?", "Help me with my financial queries"],
    "enquiry": ["I have a question about my loan", "Can you provide more details?", "I need clarification on loan policies", "Tell me about the repayment process", "What are the terms of my loan?"],
    "fallback": ["I don’t understand", "This doesn’t make sense", "Can you repeat that?", "I need more information", "Sorry, I didn’t get that"]
}

In [7]:
intent_embeddings = {key: model.encode(sentences, convert_to_tensor=True) for key, sentences in intent_examples.items()}

In [19]:
#Refresh convo.

template = """You are an intelligent virtual financial agent helping our customer {first_name} {last_name}.
            Your role is to help manage the customer's loan repayment and answer their financial questions in a clear and precise way. 

            Instructions:
            1. Use precise financial language and ensure clear, accurate information.
            2. If the user is willing to pay the loan then please provide this link '''https://paymentUSER1UDN.com'''. Do not send the link until user requests or user wants to pay the loan.
            3. If the customer is struggling, provide options like grace periods, payment restructuring, or deadline extensions considering their income, number of late repayment and loan amount yet to be repayed.
            4. Keep responses short and to the point.
            5. Ensure confidentiality and remind the customer to keep their payment details secure.
            6. You can only extend the last loan repayment date by a maximum of 10 days if user requests for grace periods or deadline extensions considering their income, number of late repayment and loan amount yet to be repayed.
            7. If the question cannot be answered using the information provided, reply with "Sorry, but I am unable to answer this query". 
            
"""

chat_history = []

In [9]:
template_messages = [
    SystemMessage(content=template),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{query}")
    #HumanMessage(content="{query}")
]

prompt_template = ChatPromptTemplate.from_messages(template_messages)

In [10]:
@tool

def get_user_data() -> dict:
    """
    Returns all information about the customer's loan.
    """
    key = int(phone)
    user_data = get_info(key)
    return user_data

In [11]:
@tool
def current_date_time() -> dict:
    '''
    Returns the current server date and time in JSON format.
    '''
    now = datetime.datetime.now()
    ist_timezone = pytz.timezone('Asia/Kolkata')
    dt_ist = now.astimezone(ist_timezone)
    time = dict()
    time['day'] = dt_ist.strftime('%A')
    time['month'] = dt_ist.strftime('%B')
    time['date'] = dt_ist.strftime('%Y-%m-%d')
    time['time'] = dt_ist.strftime('%H:%M')
    return time

In [12]:
llm_with_tools = llm.bind_tools([get_user_data, current_date_time])

tool_mapping = {
    "get_user_data" : get_user_data,
    "current_date_time" : current_date_time
}

In [13]:
def analyze_text(text):
    doc = nlp(text)

    # Extract relevant entities
    entities = {ent.label_: ent.text for ent in doc.ents if ent.label_ in ["ORG", "GPE", "PERSON", "PRODUCT", "MONEY", "DATE", "CARDINAL", "EVENT", "TIME"]}

    # Encode input text for intent matching
    input_embedding = model.encode(text, convert_to_tensor=True)

    # Find best matching intent
    best_intent = "unknown"
    best_score = 0.4  # Confidence threshold

    for intent, embeddings in intent_embeddings.items():
        similarity_score = util.pytorch_cos_sim(input_embedding, embeddings).max().item()
        if similarity_score > best_score:
            best_intent = intent
            best_score = similarity_score

    return {"entities": entities, "intent": best_intent}

In [14]:
phone = 9804604602

user_data = get_info(phone)

user_info = {
    "first_name": user_data['first_name'],
    "last_name": user_data['last_name']
}

In [15]:
formatted_template = template.format(**user_info)
template_messages[0] = SystemMessage(content=formatted_template)
prompt_template = ChatPromptTemplate.from_messages(template_messages)

In [16]:
def chat(text:str) -> str:
    messages = prompt_template.format_messages(
        chat_history=chat_history,
        query = text
    )
    chat_history.append(HumanMessage(text))
    
    response = llm_with_tools.invoke(messages)
    chat_history.append(response)
    
    if response.tool_calls:
        for tool_call in response.tool_calls:
            tool = tool_mapping[tool_call['name']]
            output = tool.invoke(tool_call['args'])
            chat_history.append(ToolMessage(str(output), tool_call_id=tool_call['id']))
            ai_says = llm_with_tools.invoke(chat_history)
        chat_history.append(ai_says)
    else:
        ai_says = response

    return ai_says.content

In [17]:
def recognize_speech():
    recognizer = sr.Recognizer()
    with sr.Microphone() as source:
        print("_"*100)
        try:
            audio = recognizer.listen(source, timeout=3000)
            user_input = recognizer.recognize_google(audio)
            print(f"You said: {user_input}")
            return user_input
        except sr.UnknownValueError:
            return None
        except sr.WaitTimeoutError:
            return None

In [20]:
welcome = AIMessage(content=f"Hello {user_info['first_name']}, I wanted to talk about your loan repayment.")
chat_history.append(welcome)
print('Bot: ',welcome.content)
speaker.speak(welcome.content)

while True:
    print('Listening')
    query = recognize_speech()
    if query is not None:
        if query.lower() in ['okay thank you','ok thank you','ok thank you']:
            #res = chat(query)
            #speaker.speak(res)
            break
        else:
            print(analyze_text(query))
            res = chat(query)
            cleaned_text = regex.sub(r'[\/@|!?]', '', res)
            print('Bot: ',cleaned_text)
            speaker.speak(cleaned_text)
            

Bot:  Hello Diya, I wanted to talk about your loan repayment.
 > Text splitted to sentences.
['Hello Diya, I wanted to talk about your loan repayment.']
 > Processing time: 2.669152021408081
 > Real-time factor: 0.6081046668049284
Listening
____________________________________________________________________________________________________
You said: hello
{'entities': {}, 'intent': 'greet'}
Bot:  Hello How can I assist you today with your loan repayment or any other financial questions
 > Text splitted to sentences.
['Hello How can I assist you today with your loan repayment or any other financial questions']
 > Processing time: 1.5998270511627197
 > Real-time factor: 0.2706889692920348
Listening
____________________________________________________________________________________________________
You said: tell me my loan type
{'entities': {}, 'intent': 'loan_details'}
Bot:  Your loan type is a Consumer Durable Loan. How else can I assist you with your loan or other financial needs
 > T

In [21]:
for message in chat_history:
    print(message.content)

Hello Diya, I wanted to talk about your loan repayment.
hello
Hello! How can I assist you today with your loan repayment or any other financial questions?
tell me my loan type

{'first_name': 'Diya', 'last_name': 'Sharma', 'phone_no': 9804604602, 'gender': 'Female', 'income_in_inr': 380418.9, 'credit_score': 808, 'loan_type': 'Consumer Durable Loan', 'loan_amount': 36100.6, 'interest_rate': 12.4, 'process_fee': 361.0, 'installment': 6236.2, 'start_date': '2024-05-09', 'tenure': 6, 'balance_to_pay': 25866.4, 'payment_mode': 'Debit Card', 'late_payment': 0, 'last_date': '2024-08-03'}
Your loan type is a Consumer Durable Loan. How else can I assist you with your loan or other financial needs?
tell me about my loan amount also
Your loan amount is INR 36,100.60. If you have any other questions or need further assistance, feel free to ask!
how much do I need to pay
You need to pay INR 25,866.40 to clear your loan. If you're ready to make a payment, let me know and I can guide you through the

In [ ]:
import os
import time
import pandas as pd
import pygame
from io import BytesIO
from gtts import gTTS
import speech_recognition as sr
from pydub import AudioSegment, silence

In [25]:

# Define a list of trigger words for interruption
TRIGGER_WORDS = ["stop", "interrupt", "wait", "hold on", "pause", "stop talking", "wait a moment"]

# Function to detect voice activity (VAD)
def vad_detect(audio):
    sound = AudioSegment.from_file(BytesIO(audio.get_wav_data()), format="wav")
    silent_chunks = silence.detect_silence(sound, min_silence_len=500, silence_thresh=-40)

    if silent_chunks:
        return True  # Detected an interruption
    return False  # No interruption

# Function to play text as speech
def speak_text(text):
    try:
        tts = gTTS(text=text, lang='en')
        tts.save("response.mp3")
        pygame.mixer.init()
        pygame.mixer.music.load("response.mp3")
        pygame.mixer.music.play()

        # Allow interruption
        while pygame.mixer.music.get_busy():
            if listen_for_input(interrupt_check=True) == "interrupt":
                pygame.mixer.music.stop()
                print("Interruption detected. Stopping speech.")
                return
            time.sleep(0.1)

    except Exception as e:
        print(f"Error: {e}")

# Function to listen for user input and interruptions
def listen_for_input(interrupt_check=False):
    recognizer = sr.Recognizer()
    with sr.Microphone() as source:
        print("Listening...")
        recognizer.adjust_for_ambient_noise(source)
        audio = recognizer.listen(source)

        if interrupt_check and vad_detect(audio):
            return "interrupt"

        try:
            response = recognizer.recognize_google(audio)
            print(f"User said: {response}")
            if any(word in response.lower() for word in TRIGGER_WORDS):
                return "interrupt"
            return response.lower()
        except sr.UnknownValueError:
            print("Sorry, I couldn't understand that.")
            return None
        except sr.RequestError:
            print("Speech recognition service error.")
            return None

# Function to fetch borrower details
def search(phone_number, file_path="Demo1.csv"):
    try:
        data = pd.read_csv(file_path)
        borrower = data[data['Phone Number'] == phone_number].to_dict(orient='records')
        return borrower[0] if borrower else None
    except FileNotFoundError:
        print(f"Error: File {file_path} not found.")
        return None

# Call agent function with VAD-enabled interruptions
def initiate_call(phone_number):
    borrower_data = search(phone_number)
    if not borrower_data:
        print(f"No data found for phone number: {phone_number}.")
        return

    agent_name = "Malti"
    company_name = "Predixion AI"
    borrower_name = borrower_data.get("Borrower Name", "Valued Customer")

    # Agent introduction
    intro = f"Hi, I’m {agent_name} from {company_name}. Could you please tell me your first name?"
    print("Bot:", intro)
    speak_text(intro)

    while True:
        user_response = listen_for_input()
        if user_response == "interrupt":
            print("Bot: I'll get back to you when you're ready.")
            speak_text("I'll get back to you when you're ready.")
            return
        elif user_response:
            customer_name = user_response
            print(f"Bot: Am I speaking with {customer_name}?")
            speak_text(f"Am I speaking with {customer_name}?")
            break

    # Check if the name exists in the database
    matched_data = search(phone_number)
    if matched_data:
        balance = matched_data.get("Outstanding Balance", "N/A")
        due_date = matched_data.get("Due Date", "N/A")

        loan_info = f"Your outstanding loan balance is ₹{balance}. Your due date for repayment is {due_date}."
        print("Bot:", loan_info)
        speak_text(loan_info)
    else:
        print("Bot: Sorry, I couldn't find your details. Please try again.")
        speak_text("Sorry, I couldn't find your details. Please try again.")

if _name_ == "_main_":
    phone_number = 9324396175  
    initiate_call(phone_number)

NameError: name '_name_' is not defined